<a href="https://colab.research.google.com/github/wlgns222/ROKA/blob/main/ai-study/deep-learning-from-scratch-vol1/Ch7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ch7. CNN

**합성곱 신경망(Convolutional Nerual Network)** 은 이미지 인식, 음성 인식 등 다양한 곳에서 사용된다.

특히 이미지 인식 분야에서 딥러닝을 활용한 기법은 거의 CNN을 기초로 한다.

## 7.1 전체 구조

지금까지 본 신경망은 인접하는 계층의 모든 뉴런과 결합되어 있다.

이를 **완전연결**이라 하며, 완전히 연결된 계층을 **Affine 계층**이라는 이름으로 구현했다.

<br>

**완전연결 계층**

Input -> 'Affine - ReLU' -> 'Affine - ReLU' -> 'Affine - ReLU' -> 'Affine - Softmax'

CNN에는 **합성곱 계층, 풀링 계층**이 추가된다.

CNN 계층은 'Conv - ReLU - (Pooling)' 흐름으로 연결된다.

<br>

**CNN 신경망 예시**

Input -> 'Conv - ReLU - Pooling' -> 'Conv - ReLU' -> 'Affine - ReLU' -> 'Affine - Softmax'

출력에 가까운 층에서는 'Affine - ReLU' 조합을 사용하며, 출력 계층에서는 'Affine - Softmax'을 사용함을 볼 수 있다.

## 7.2 합성곱 계층

### 7.2.1 완전연결 계층의 문제점

완전연결 계층에서는 인접하는 계층의 뉴런이 모두 연결되고 출력 수는 임의로 정할 수 있다.


**완전연결 계층의 문제점 - 데이터 형상이 무시된다**

이전까지는 형상이 (1, 28, 28) 인 이미지를 1줄로 세운 784 데이터를 첫 Affine 계층에 입력했다.

해당 방식은 이미지를 1차원으로 늘어뜨려 이미지가 가지고 있는 2차원 또는 3차원 형상의 공간적 정보와 본질적인 패턴에 대한 정보를 살릴 수 없다.

합성곱 계층은 형상을 유지한다. 이미지도 3차원 데이터로 입력받으며, 마찬가지로 다음 계층에도 3차원 데이터를 전달한다.

따라서 CNN에서는 이미지처럼 형상을 가진 데이터를 이해할 능력이 생긴다.

CNN 에서는 입출력 데이터를 **특징맵** 이라 하며, 입력 데이터를  **입력 특징맵**, 출력 데이터를 **출력 특징맵**이라고 한다.

### 7.2.2 합성곱 연산

**필터(또는 커널) 이란**

합성곱 연산에서 입력되는 필터로 형상을 (높이, 너비)로 표기한다.

**합성곱 연산**은 필터의 윈도우를 일정 간격으로 이동해가며 입력 데이터에 적용한다.

필터가 이미지를 흝으면서 대응하는 요소끼리 곱하고 그 합을 구한다. (해당 계산을 **단일 곱셈-누산(FMA)** 라고 한다.)

**완전연결 신경망**에서는 가중치 매개변수와 편향이 존재하는데,

**CNN**에서는 필터의 매개변수가 그동안의 '가중치'에 해당한다. 편향은 언제나 1x1 하나만 존재하며, 그 하나의 값을 필터를 적용한 모든 원소에 더한다.

### 7.2.3 Fadding

합성곱을 하면 출력의 크기가 줄어든다. 예를 들어 (4,4) 입력 데이터에 (3,3) 필터를 적용할 시 출력은 (2,2) 가 되어 입력보다 2만큼 줄어든다. 이는 합성곱 연산을 몇번이나 되풀이하는 신경망에서는 문제가 된다.

이를 방지하기 위한 기법이 **패딩(Fadding)** 이다.

**패딩이란**

합성곱 연산을 수행하기 전에 입력 데이터 주변에 특정 더미 값 (주로 0) 을 채우는 것이다.

패딩을 사용함으로써 입력과 동일한 크기의 출력을 얻을 수 있으며, 외곽 지역의 정보도 골고루 활용할 수 있다.

### 7.2.4 Stride

필터를 적용하는 위치의 간격을 **스트라이드**라고 한다.

즉, 스트라이드를 2로 하면 필터를 적용하는 윈도우가 두 칸씩 이동하게 된다. 스트라이드 값이 커지면 커질수록 출력 데이터의 크기는 줄어들고, 연산량이 감소하여 정보를 거시적으로 보게된다.

**출력 크기를 계산하는 공식**

$$OH = \frac{H + 2P - FH}{S} + 1$$

$$OW = \frac{W + 2P - FW}{S} + 1$$

- $H, W$ : 입력 크기 (높이/너비) / $FH, FW$ : 필터 크기 (높이/너비) / $OH, OW$ : 출력 크기 (높이/너비)
- $P$ : 패딩 / $S$ : 스트라이드

출력 크기를 계산하는 이 공식은 CNN 설계의 기본 규격이다.

분수 값이 나오면 연산이 불가능하므로, 보통 패딩과 스트라이드를 조절해 정수가 나오도록 설계를 하거나 반올림하도록 설계한다.

### 7.2.5 3차원 데이터 합성곱 연산

3차원 데이터의 경우 기존 세로, 가로에 **채널**이 포함된다.

채널 쪽으로 특징 맵이 여러개 있다면 입력 데이터와 필터의 합성곱 연산을 채널마다 수행하고 결과를 더해서 하나의 출력을 얻는다.

**이때 필터의 채널 수는 입력 데이터의 채널 수와 같아야 한다.**

### 7.2.6 블록으로 생각하기

3차원 데이터의 형상을 (채널, 높이, 너비)로 나타낼 수 있다.

입력 데이터 (C, H, W) 와 필터 (C, FH, FW) 에 대한 합성곱 연산을 수행하면 (1, OH, OW) 즉, **한장의 출력 맵**이 나온다.

합성곱 연산의 출력으로 다수의 채널을 내보내기 위해서는 **필터(가중치)를 여럿 사용**하면 된다.


(C,H,W) 와 FN 개의 (C, FH, FW) 를 연산하게 되면 (FN, OH, OW) 의 출력 데이터를 얻게된다.

그런 이유에서 **필터의 가중치 데이터는 4차원 데이터**이다. - (출력 채널 수, 입력 채널 수, 높이, 너비)

## 7.3 Pooling Layer

**풀링은 세로, 가로 방향 공간을 줄이는 연산이다**

원리 : 윈도우를 움직이며 해당 윈도우에서 특정 값을 꺼낸다.

풀링에는 최대 풀링과 평균 풀링이 있다.

**최대 풀링**
- 대상 영역에서 가장 큰 원소를 꺼낸다.
- 가장 강한 신호만 추출하는 방식이다.

**평균 풀링**
- 대상 영역에서 평균을 계산한다.
- 정보를 너무 뭉뚱그리는 경향이 있어, 현대 시각 지능에서는 잘 쓰이지 않는다.

### 7.3.1 풀링 계층의 특징

**학습해야 할 매개변수가 없다.**

풀링 계층은 합성곱과는 달리 학습해야 할 매개변수가 없다. 풀링은 대상 영역에서 최댓값이나 평균을 구하는 간단 명확한 처리이므로 학습이 불필요하다.

**채널 수가 변하지 않는다.**

채널마다 독립적으로 계산하기 때문에 풀링 계층은 입력 데이터의 채널 수를 출력 데이터에 그대로 보낸다.

**입력의 변화에 영향을 적게 받는다.**

풀링은 데이터의 작은 변화에 영향을 적게 받는다.

## 7.4 합성곱/풀링 계층 구현

### 7.4.1 4차원 배열

CNN 에서 계층 사이를 흐르는 데이터는 4차원이다.

예를 들어 데이터의 형상이 (10, 1, 28, 28) 일 때, 이는 높이 28, 너비 28, 채널 1개인 데이터가 10개라는 뜻이다.

따라서 합성곱 연산의 구현은 복잡할 것 같지만 **im2col** 이라는 트릭을 이용하여 문제를 단순히 만들 수 있다.

### 7.4.2 im2col 로 데이터 전개하기

**im2col**은 입력 데이터를 필터링(가중치 계산)하기 좋게 전개하는(펼치는) 함수이다.


즉, 복잡한 **4차원의 합성곱 연산을 행렬 연산**으로 바꾸어 준다.

**im2col 방식**

**입력 데이터에서 필터를 적용 영역을 한 줄로 늘어놓는다.**

일반적으로 필터링하는 영역이 겹치는 경우가 대부분이기 때문에 im2col 로 전개한 후 원소 수가 원래 블록의 원소 수보다 많아진다.

im2col 을 사용하면 메모리를 더 많이 소비한다는 단점이 있다. 그러나 현대 연산 장치는 행렬 계산을 하는데 탁월하다. 따라서 효율을 높일 수가 있다.

**합성곱 연산의 상세 과정**

필터를 세로 1열로 전개하고, im2col이 전개한 데이터와 행렬 곱을 계산한다. 마지막으로 출력 데이터를 변형(reshape)한다.

### 7.4.3 합성곱 계층 구현하기

**im2col 함수의 인터페이스**


```
im2col(input_data, filter_h, filter_w, stride=1, pad=0)
```
**input_data : (데이터의 수, 채널수, 높이, 너비) 의 4차원 배열로 이루어진 입력 데이터**

filter_h / filter_w : 필터의 높이/너비

stride / pad : 스트라이드 / 패딩

> 책에서는 im2col 함수를 미리 만들어 제공한다.

In [2]:
import numpy as np

In [8]:
!git clone https://github.com/WegraLee/deep-learning-from-scratch.git book_code

import sys, os
sys.path.append('book_code')
from common.util import im2col

from collections import OrderedDict
from common.layers import Relu, Affine, SoftmaxWithLoss

x1 = np.random.rand(1, 3, 7, 7)
col1 = im2col(x1, 5, 5, stride = 1, pad = 0)
print(col1.shape) #(9, 75)

x1 = np.random.rand(10, 3, 7, 7)
col1 = im2col(x1, 5, 5, stride = 1, pad = 0)
print(col1.shape) #(90, 75)

# 배치의 크기가 1에서 10으로 10배 증가하면, im2col의 첫번째 차원의 결과도 10배 증가한다.

fatal: destination path 'book_code' already exists and is not an empty directory.
(9, 75)
(90, 75)


**합성곱 계층 - class Convolution**

In [4]:
class Convolution:
  def __init__(self, W, b, stride=1, pad=0):
    self.W = W
    self.b = b
    self.stride = stride
    self.pad = pad

  def forward(self, x) :
    FN, C, FH, FW = self.W.shape
    N, C, H, W = x.shape
    out_h = int(1 + (H + 2*self.pad - FH) / self.stride)
    out_w = int(1 + (W + 2*self.pad - FW) / self.stride)

    #Unroll input with im2col
    col = im2col(x, FH, FW, self.stride, self.pad)
    #Unroll filter
    col_W = self.W.reshape(FN, -1).T
    #Matrix Multiplication + Bias
    out = np.dot(col, col_W) + self.b

    #Reshape basck to 4D (N, H, W, C) -> (N, C, H, W)
    out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)

    return out


### 7.4.4 풀링 계층 구현하기

**풀링 계층 - class Pooling**

In [5]:
class Pooling :
  def __init__(self, pool_h, pool_w, stride=2, pad=0) :
    self.pool_h = pool_h
    self.pool_w = pool_w
    self.stride = stride
    self.pad = pad

  def forward (self, x) :
    N, C, H, W = x.shape
    out_h = int(1 + (H - self.pool_h) / self.stride)
    out_w = int(1 + (W - self.pool_w) / self.stride)

    #Unroll input with im2col
    col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
    #Reshape so that each pooling window forms a row
    col = col.reshape(-1, self.pool_h * self.pool_w)
    #Pick the strongest signal
    out = np.max(col, axis = 1)
    #Reshape basck to 4D (N, H, W, C) -> (N, C, H, W)
    out = out.reshape(N, out_h, out_w, C).transpose(0, 3, 1, 2)

    return out


## 7.5 CNN 구현

단순한 CNN 신경망의 구성

input -> (Conv->ReLU->Pooling) -> (Affine->ReLU) -> (Affine->Softmax) ->

**class SimpleConvNet**

초기화할 때 받는 인수
- input_dim : 입력데이터(채널 수, 높이, 너비)의 차원
- conv_param : 합성곱 계층의 하이퍼파라미터, 딕셔너리의 키는 다음과 같다.
  - filter_num - 필터 수
  - filter_size - 필터 크기
  - stride - 스트라이드
  - pad - 패딩
  - hidden_size - 은닉층의 뉴런 수
  - output_size - 출력층의 뉴런 수
  - weight_init_std - 초기화 때의 가중치 표준편차



In [9]:
from collections import OrderedDict
from common.layers import Convolution, Relu, Pooling, Affine, SoftmaxWithLoss

In [12]:
class SimpleConvNet:
    def __init__(self, input_dim=(1, 28, 28),
                 conv_param={'filter_num':30, 'filter_size':5, 'pad':0, 'stride':1},
                 hidden_size=100, output_size=10, weight_init_std=0.01):

        # 1. Extract hyperparameters
        filter_num = conv_param['filter_num']
        filter_size = conv_param['filter_size']
        filter_pad = conv_param['pad']
        filter_stride = conv_param['stride']
        input_size = input_dim[1]

        # 2. Calculate output size of Conv/Pool layers
        conv_output_size = (input_size + 2*filter_pad - filter_size) // filter_stride + 1
        pool_output_size = int(filter_num * (conv_output_size/2) * (conv_output_size/2))

        # 3. Initialize weights (W1 = Conv filters, W2/W3 = Affine weights)
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(filter_num, input_dim[0], filter_size, filter_size)
        self.params['b1'] = np.zeros(filter_num)
        self.params['W2'] = weight_init_std * np.random.randn(pool_output_size, hidden_size)
        self.params['b2'] = np.zeros(hidden_size)
        self.params['W3'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b3'] = np.zeros(output_size)

        # Create Layers
        self.layers = OrderedDict()
        self.layers['Conv1'] = Convolution(self.params['W1'], self.params['b1'], conv_param['stride'], conv_param['pad'])
        self.layers['Relu1'] = Relu()
        self.layers['Pool1'] = Pooling(pool_h=2, pool_w=2, stride=2)
        self.layers['Affine1'] = Affine(self.params['W2'], self.params['b2'])
        self.layers['Relu2'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W3'], self.params['b3'])

        self.last_layer = SoftmaxWithLoss()


**predict & loss : 추론과 손실함수**

In [13]:
def predict(self, x):
  for layer in self.layers.values():
    x = layer.forward(x)
  return x

def loss(self, x, t):
  y = self.predict(x)
  return self.last_layer.forward(y, t)

**gradient : 오차역전파법**

In [14]:
def gradient(self, x, t) :
  self.loss(x, t)

  dout = 1
  dout = self.last_layer.backward(dout)

  layers = list(self.layers.values())
  layers.reverse()

  for layer in layers:
    dout = layer.backward(dout)

  grads = {}
  grads['W1'] = self.layers['Conv1'].dW
  grads['b1'] = self.layers['Conv1'].db
  grads['W2'] = self.layers['Affine1'].dW
  grads['b2'] = self.layers['Affine1'].db
  grads['W3'] = self.layers['Affine2'].dW
  grads['b3'] = self.layers['Affine2'].db

  return grads

## 7.6 CNN 시각화

### 7.6.1 첫 번째 층 가중치를 시각화

첫 번째 층의 필터를 시각화해보면 모델이 무엇을 보는지 알 수 있다.

학습 전 : 무작위 노이즈와 같은 형태

**학습 후 : 규칙성이 있는 이미지**
- 에지 : 가로 또는 세로선, 색상이 바뀌는 경계선
- 블롭 : 덩어리, 국소적으로 덩어리진 형태

### 7.6.2 층 깊이에 따른 추출 정보 변화

**합성곱 계층을 여러 층 쌓으면, 층이 깊어질 수록 더 복잡하고 추상화된 정보가 추출된다.**

초반 층 : 에지와 블롭 & 텍스처

중반 층 : 원과 사각형같은 사물의 일부

후반 층 : 개, 자동차와 같은 사물을 인식



## 7.7 대표적인 CNN

### 7.7.1 LeNet

LeNet 은 손글씨 숫자를 인식하는 신경망으로 1988년에 제안되었다.

합성곱 계층과 풀링 계층을 반복하고, 마지막으로 완전 연결 계층을 거치면서 결과를 출력한다.

**현대 CNN 과의 차이점**

LeNet 은 시그모이드 함수를 사용하는데 반해 현재는 주로 ReLU 를 사용한다.

LeNet 은 서브샘플링을 통해 중간 데이터의 크기를 줄이지만, 현재는 최대 풀링이 주류이다.

**LeNet 은 1988년에 제안된 첫 CNN 이다.**

### 7.7.2 AlexNet

2012년에 발표된 AlexNet 은 딥러닝 열풍에 큰 역할을 하였다.

**LeNet 과의 차이점**

- 활성화 함수로 ReLU 를 사용한다.
- LRN 이라는 국소적 정규화를 실시하는 계층을 이용한다.
- 드롭아웃을 사용한다.

구성 면에서 LeNet 와 AlexNet에 큰 차이는 없다.

그러나 이를 둘러싼 환경과 컴퓨터 기술이 큰 진보를 이루었다. 대량의 데이터를 누구나 얻을 수 있게되고, 병렬 계산에 특화된 GPU가 보급되며 대량의 연산을 고속으로 수행할 수 있게 되었다.